# Phase 4 — Saving Your Work to Disk and Loading It Back

Everything so far has lived in the notebook's memory, which vanishes the moment you restart
the kernel. Sooner or later you'll compute something you don't want to lose — cleaned-up
data that took ten minutes to prepare, or a trained model's weights — and you'll need it on
disk.

NumPy gives you three ways to do that. Picking the wrong one either loses information or
wastes a lot of space, so here's the whole menu up front:

| What to call | File you get | Squashed smaller? | Holds | Reach for it when |
|---|---|---|---|---|
| `np.save` / `np.load` | `.npy` | no | **one** array | Your default. Fast and exact. |
| `np.savez` / `np.load` | `.npz` | no | **several** named arrays | Keeping related arrays in one file. |
| `np.savez_compressed` | `.npz` | **yes** | several named arrays | Big or repetitive data. Slower to write. |
| `np.savetxt` / `np.loadtxt` | `.csv`, `.txt` | no | one flat or table-shaped array | A human, Excel, or another language has to open it. |

### The one rule that decides it

**Binary files (`.npy`, `.npz`) give you back exactly what you put in.** Same numbers, same
type, same shape.

**Text files (`.csv`) don't.** Everything comes back as a decimal even if you saved whole
numbers, and anything more than a simple table can't be saved at all. Text is for *sharing*,
not for storing.

### What this notebook covers
1. Saving one array
2. Saving several at once
3. Saving as text for other people
4. Two things that will bite you

In [1]:
# ==========================================
# SETUP
# ==========================================
import numpy as np
import os                    # for checking file sizes and tidying up afterwards

## 1. Saving one array — `.npy`

`np.save` writes the raw numbers plus a small note recording the shape and type. Because
that note travels with the data, what you load back is identical to what you saved — no
guessing, no conversion.

If you leave the `.npy` off the filename, NumPy adds it for you.

In [2]:
# ==========================================
# 1a. SAVING
# ==========================================
array1 = np.array([1, 2, 3, 4, 5])                      # whole numbers
array2 = np.random.default_rng(0).random((3, 3))        # random decimals
array3 = np.zeros((3, 3))                               # all zeros

# Each file holds exactly one array
np.save('array1.npy', array1)
np.save('array2.npy', array2)
np.save('array3.npy', array3)

print("files written:", [f for f in sorted(os.listdir('.')) if f.startswith('array')])

files written: ['array1.npy', 'array2.npy', 'array3.npy']


In [3]:
# ==========================================
# 1b. LOADING IT BACK AND CHECKING NOTHING CHANGED
# ==========================================
loaded1 = np.load('array1.npy')
loaded2 = np.load('array2.npy')

print("array1 came back as:", loaded1)
print("array2 came back as:\n", loaded2.round(4))

# The round trip is exact — all three of these should say True
print("\nsame numbers?", np.array_equal(array1, loaded1))
print("same type?   ", array1.dtype == loaded1.dtype)
print("same shape?  ", array2.shape == loaded2.shape)

array1 came back as: [1 2 3 4 5]
array2 came back as:
 [[0.637  0.2698 0.041 ]
 [0.0165 0.8133 0.9128]
 [0.6066 0.7295 0.5436]]

same numbers? True
same type?    True
same shape?   True


In [4]:
# ==========================================
# 1c. LOADING A FILE SAVED IN AN EARLIER SESSION
# ==========================================
# 3DSpiral.npy was saved some time ago. Note that we don't have to tell np.load
# anything about it — the shape and type were stored inside the file.
spiral = np.load('3DSpiral.npy')

print("shape   :", spiral.shape, "→ 100 points, each with an x, y and z")
print("type    :", spiral.dtype)
print("first 3 points:\n", spiral[:3].round(4))
print("\nhow far the x values spread:",
      spiral[:, 0].min().round(2), "to", spiral[:, 0].max().round(2))

shape   : (100, 3) → 100 points, each with an x, y and z
type    : float64
first 3 points:
 [[-0.4097 -0.7092  3.6363]
 [ 0.2692  1.2007  0.157 ]
 [ 0.6616  0.7833  0.9411]]

how far the x values spread: -1.06 to 1.22


## 2. Saving several arrays at once — `.npz`

An `.npz` file is really just a zip folder with one `.npy` inside it per array.

You pass your arrays in as named arguments — `scores=scores` — and those names become the
keys you use to fetch them back out. Much better than three loose files and a memory of
which was which.

**Use `with np.load(...) as data:`** rather than plain `np.load(...)`. An `.npz` doesn't read
everything immediately; it keeps the file open and fetches each array only when you ask for
it. The `with` block guarantees the file gets closed when you're finished.

In [5]:
# ==========================================
# 2. BUNDLING ARRAYS INTO ONE FILE
# ==========================================
scores = np.array([85, 90, 78, 92, 88])
grades = np.array(['B', 'A', 'C', 'A', 'B'])
names  = np.array(['Ana', 'Ben', 'Cy', 'Dee', 'Eli'])

# The names on the left of each = become the keys we look them up by later
np.savez('student_data.npz', scores=scores, grades=grades, names=names)

# 'with' makes sure the file is properly closed when we're done with it
with np.load('student_data.npz') as data:
    print("what's inside :", list(data.keys()))
    print("scores        :", data['scores'])
    print("grades        :", data['grades'])
    print("names         :", data['names'])
    print("\ngrades type :", data['grades'].dtype,
          "← text, stored exactly as it was. No conversion happened.")

what's inside : ['scores', 'grades', 'names']
scores        : [85 90 78 92 88]
grades        : ['B' 'A' 'C' 'A' 'B']
names         : ['Ana' 'Ben' 'Cy' 'Dee' 'Eli']

grades type : <U1 ← text, stored exactly as it was. No conversion happened.


### `savez` vs `savez_compressed` — and a common misconception

**`np.savez` does not compress anything.** It's a zip *container*, but the contents are
stored as-is. `np.savez_compressed` is the one that actually squashes them.

How much you gain depends entirely on how repetitive your data is, and the difference is
dramatic:

- **Repetitive data** (all zeros, lots of repeated labels) squashes enormously — "500 rows
  of zeros" is a very short thing to write down.
- **Random data** barely budges. There's no pattern to exploit, so there's nothing to
  shorten.

You pay for compression in time, both saving and loading. For big, repetitive arrays it's
well worth it. Let's measure both cases.

In [6]:
# ==========================================
# 2b. HOW MUCH DOES COMPRESSION ACTUALLY SAVE?
# ==========================================
repetitive = np.zeros((500, 500))                          # nothing but zeros
random_arr = np.random.default_rng(1).random((500, 500))   # no pattern whatsoever

np.savez('demo_plain.npz',          rep=repetitive, rnd=random_arr)   # no squashing
np.savez_compressed('demo_zip.npz', rep=repetitive, rnd=random_arr)   # squashed

plain = os.path.getsize('demo_plain.npz') / 1024      # bytes → kilobytes
comp  = os.path.getsize('demo_zip.npz')   / 1024
print(f"savez            : {plain:8.1f} KB")
print(f"savez_compressed : {comp:8.1f} KB  → {plain / comp:.1f}x smaller")
print("(only middling, because half of this file is unsquashable random data)")

# Now the same test on the zeros ALONE, where compression really shows off
np.savez('zeros_plain.npz',          a=repetitive)
np.savez_compressed('zeros_zip.npz', a=repetitive)
p = os.path.getsize('zeros_plain.npz') / 1024
c = os.path.getsize('zeros_zip.npz')   / 1024
print(f"\njust the zeros: {p:.1f} KB → {c:.1f} KB  ({p / c:.0f}x smaller!)")
print("('250,000 zeros' is a very short sentence to write down.)")

savez            :   3906.7 KB
savez_compressed :   1844.4 KB  → 2.1x smaller
(only middling, because half of this file is unsquashable random data)

just the zeros: 1953.4 KB → 2.1 KB  (923x smaller!)
('250,000 zeros' is a very short sentence to write down.)


## 3. Saving as text for other people

Use this only when something outside Python has to read the file — a colleague with Excel,
an R script, a web app. `savetxt` handles flat rows and simple tables only; a 3-D array has
no obvious way to become a spreadsheet, so it refuses.

The arguments worth knowing:

- **`delimiter=','`** — separate with commas, making it a proper CSV. Without this you get
  spaces.
- **`header='id,score'` plus `comments=''`** — writes a real header row. Leave `comments`
  out and NumPy puts a `#` in front of your header, which then confuses every spreadsheet
  program that opens it.
- **`fmt='%d'`** — how to write each number. `%d` means whole numbers, `%.2f` means two
  decimal places. Skip it and you get NumPy's default scientific notation
  (`2.392500000000e+04`), which is technically correct and horrible to read.

In [7]:
# ==========================================
# 3. WRITING A CSV ANYONE CAN OPEN
# ==========================================
data = np.array([[1, 85, 23925],
                 [2, 90, 27197],
                 [3, 78, 25637]])

np.savetxt('results.csv', data,
           delimiter=',',                  # commas between values
           header='id,score,sales',        # a real header line
           comments='',                    # ...without a '#' glued to the front of it
           fmt='%d')                       # write them as whole numbers

print("here's the actual file on disk:")
with open('results.csv') as fh:
    print(fh.read())

# Reading it back. skiprows=1 means "ignore the header line, it's not data"
loaded_csv = np.loadtxt('results.csv', delimiter=',', skiprows=1)
print("loaded back in:\n", loaded_csv)
print("\ntype:", loaded_csv.dtype,
      "← everything is a decimal now, even the ids. Text files can't remember types.")
print("cast back to whole numbers:\n", loaded_csv.astype(int))

here's the actual file on disk:
id,score,sales
1,85,23925
2,90,27197
3,78,25637

loaded back in:
 [[1.0000e+00 8.5000e+01 2.3925e+04]
 [2.0000e+00 9.0000e+01 2.7197e+04]
 [3.0000e+00 7.8000e+01 2.5637e+04]]

type: float64 ← everything is a decimal now, even the ids. Text files can't remember types.
cast back to whole numbers:
 [[    1    85 23925]
 [    2    90 27197]
 [    3    78 25637]]


### When the file has gaps: `genfromtxt`

Real CSVs have holes in them — a sensor that dropped out, a form field nobody filled in.

`np.loadtxt` gives up entirely when it hits an empty cell. `np.genfromtxt` is slower but
handles it: gaps become **`nan`**, which stands for "not a number" and acts as a marker
meaning *"there was supposed to be a value here and there wasn't."*

**Be careful with `nan`: it's contagious.** Anything you calculate with it also becomes
`nan`, so a single missing cell can turn your whole average into `nan` with no error
message. The `nan`-aware functions (`np.nanmean`, `np.nansum`, …) skip the gaps instead.

In [8]:
# ==========================================
# 3b. HANDLING MISSING VALUES
# ==========================================
# Write a deliberately messy file: row 2 has no score, row 3 has no sales figure
with open('messy.csv', 'w') as fh:
    fh.write("id,score,sales\n1,85,23925\n2,,27197\n3,78,\n")

messy = np.genfromtxt('messy.csv', delimiter=',', skip_header=1)
print("loaded with the gaps marked as nan:\n", messy)

print("\nwhere are the holes:\n", np.isnan(messy))       # True wherever a value is missing
print("\nhow many holes:", np.isnan(messy).sum())
print("\nfilling the holes with 0:\n", np.nan_to_num(messy))

# The contagion problem, and the cure:
print("\nnp.mean of the score column   :", np.mean(messy[:, 1]),
      "← one missing value ruined the whole answer")
print("np.nanmean of the score column:", np.nanmean(messy[:, 1]),
      "← this one just steps over the gap")

loaded with the gaps marked as nan:
 [[1.0000e+00 8.5000e+01 2.3925e+04]
 [2.0000e+00        nan 2.7197e+04]
 [3.0000e+00 7.8000e+01        nan]]

where are the holes:
 [[False False False]
 [False  True False]
 [False False  True]]

how many holes: 2

filling the holes with 0:
 [[1.0000e+00 8.5000e+01 2.3925e+04]
 [2.0000e+00 0.0000e+00 2.7197e+04]
 [3.0000e+00 7.8000e+01 0.0000e+00]]

np.mean of the score column   : nan ← one missing value ruined the whole answer
np.nanmean of the score column: 81.5 ← this one just steps over the gap


## 4. Two things that will bite you

### `allow_pickle` is a security setting, not a convenience flag

If an array holds actual Python objects (dictionaries, lists, anything that isn't a plain
number), NumPy can't store it as raw numbers. It falls back on **pickling** — Python's
"turn any object into bytes" format.

The catch: **un-pickling a file runs code from inside it.** A malicious `.npy` can do
anything your Python process can. That's why NumPy refuses by default and makes you ask.

**Only ever pass `allow_pickle=True` for a file you created yourself.** Treat a downloaded
`.npy` the same way you'd treat a stranger's `.exe`.

(Note: plain text arrays like `['A', 'B']` are *not* affected — those are stored as proper
fixed-width text and need no special permission. It's only genuine object arrays.)

### File paths are relative to where the notebook lives

`np.save('x.npy')` drops the file next to your notebook, not wherever you happened to be.
Print `os.getcwd()` any time you can't find something you just saved.

In [9]:
# ==========================================
# 4. THE PICKLE WARNING, DEMONSTRATED
# ==========================================
# An array holding a dictionary and a list — not numbers, so this needs pickling
obj_array = np.array([{'a': 1}, [1, 2, 3]], dtype=object)
np.save('objects.npy', obj_array, allow_pickle=True)     # we must opt in to SAVE it too

# Trying to load it without permission:
try:
    np.load('objects.npy')
except ValueError as e:
    print("NumPy refused, and it's right to:", str(e)[:90], "...")

# With permission — fine here, because we made this file ourselves 30 seconds ago
print("\nwith allow_pickle=True:", np.load('objects.npy', allow_pickle=True))

print("\nfiles are being written into:", os.getcwd())

NumPy refused, and it's right to: Object arrays cannot be loaded when allow_pickle=False ...

with allow_pickle=True: [{'a': 1} list([1, 2, 3])]

files are being written into: /Users/naveen/Desktop/python/AiEngineerTutorial/numpy


In [10]:
# ==========================================
# TIDY UP — delete the demo files, keep the useful ones
# ==========================================
for f in ['demo_plain.npz', 'demo_zip.npz', 'zeros_plain.npz', 'zeros_zip.npz',
          'messy.csv', 'objects.npy']:
    if os.path.exists(f):           # check before deleting, so re-running is safe
        os.remove(f)

print("still here:", sorted(f for f in os.listdir('.')
                            if f.endswith(('.npy', '.npz', '.csv'))))

still here: ['3DSpiral.npy', 'array1.npy', 'array2.npy', 'array3.npy', 'results.csv', 'student_data.npz']


## Cheat sheet

| What you want | What to type |
|---|---|
| Save one array | `np.save('a.npy', arr)` |
| Load one array | `arr = np.load('a.npy')` |
| Save several, with names | `np.savez('f.npz', x=a, y=b)` |
| Save several, smaller | `np.savez_compressed('f.npz', x=a, y=b)` |
| Load several | `with np.load('f.npz') as d: d['x']` |
| Export for humans | `np.savetxt('f.csv', a, delimiter=',', header='...', comments='', fmt='%d')` |
| Read a text file | `np.loadtxt(..., skiprows=1)`, or `np.genfromtxt` if it has gaps |
| Deal with gaps | `np.isnan`, `np.nan_to_num`, `np.nanmean` |

### The three things worth memorising

1. **Binary keeps your types and shapes; text doesn't.** A CSV always comes back as
   decimals.
2. **`np.savez` doesn't compress.** `np.savez_compressed` does.
3. **`allow_pickle=True` runs code from the file.** Your own files only.

### Practice

1. Save the `3DSpiral` data as a CSV with 4 decimal places, load it back, and check how much
   precision you lost with `np.abs(original - reloaded).max()`.
2. Bundle `scores`, `grades` and `names` into a compressed `.npz` and compare the file size
   against the uncompressed one.
3. Save a `float32` array, load it, and check the type survived. Then do the same thing
   through `savetxt`/`loadtxt` and watch it quietly turn into `float64`.

In [11]:
# ==========================================
# PRACTICE 3 — worked solution
# ==========================================
f32 = np.array([1.5, 2.5, 3.5], dtype=np.float32)      # float32 = the smaller decimal type

# Route 1: binary. The type is written into the file, so it comes back intact.
np.save('tmp_f32.npy', f32)
print("saved as .npy →", np.load('tmp_f32.npy').dtype, "← unchanged, as promised")

# Route 2: text. The file just contains "1.5", with no record of how it was stored,
# so NumPy falls back on its default decimal type when reading it in.
np.savetxt('tmp_f32.csv', f32)
print("saved as .csv →", np.loadtxt('tmp_f32.csv').dtype, "← quietly upgraded, twice the memory")

os.remove('tmp_f32.npy'); os.remove('tmp_f32.csv')     # tidy up after ourselves

saved as .npy → float32 ← unchanged, as promised
saved as .csv → float64 ← quietly upgraded, twice the memory
